# features

> Waveform preparation and CTC decoding for the wav2vec 2.0 checkpoint.
> 
> The frontend is the Hugging Face `Wav2Vec2FeatureExtractor` path for the
> base model: 16 kHz mono, each clip normalized to zero mean and unit
> variance, then zero-padded on the right to the longest clip. Decoding is
> greedy CTC over the character vocabulary.


In [ ]:
#| default_exp features


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import json
import wave
from pathlib import Path

import numpy as np

from nada.config import Wav2Vec2Config
from nada.core import repo_root


In [ ]:
#| export
def read_wav(path: str | Path, sample_rate: int = 16_000) -> np.ndarray:
    """Read a 16-bit PCM WAV file as mono float32 in `[-1, 1]`.

    The file must already be at `sample_rate`. Resample it first if it is not.
    """
    with wave.open(str(path), "rb") as handle:
        rate, width, channels = handle.getframerate(), handle.getsampwidth(), handle.getnchannels()
        frames = handle.readframes(handle.getnframes())
    if width != 2:
        raise ValueError(f"{path}: expected 16-bit PCM, got {8 * width}-bit")
    if rate != sample_rate:
        raise ValueError(f"{path}: expected {sample_rate} Hz, got {rate} Hz. Resample it first.")
    samples = np.frombuffer(frames, dtype="<i2").astype(np.float32) / 32768.0
    return samples.reshape(-1, channels).mean(axis=1)


In [ ]:
#| export
def normalize(waveform: np.ndarray, eps: float = 1e-7) -> np.ndarray:
    """Zero mean, unit variance over the whole clip."""
    wave = np.asarray(waveform, dtype=np.float32).reshape(-1)
    return ((wave - wave.mean()) / np.sqrt(wave.var() + eps)).astype(np.float32)


In [ ]:
#| export
def prepare(waveforms: list[np.ndarray] | np.ndarray, config: Wav2Vec2Config | None = None) -> np.ndarray:
    """Normalize each clip, pad to the longest with zeros, and stack as `(batch, samples)`.

    The base checkpoint takes no attention mask, so padding is plain zeros after
    normalization, as in Hugging Face. Batch clips of similar length.
    """
    config = config or Wav2Vec2Config()
    if isinstance(waveforms, np.ndarray) and waveforms.ndim == 1:
        waveforms = [waveforms]
    clips = [normalize(wave) for wave in waveforms]
    longest = max(clip.size for clip in clips)
    config.output_length(longest)
    batch = np.zeros((len(clips), longest), dtype=np.float32)
    for index, clip in enumerate(clips):
        batch[index, : clip.size] = clip
    return batch


In [ ]:
#| export
def load_vocab(path: Path | None = None) -> dict[int, str]:
    """Id to token, from `models/wav2vec2-base-960h/vocab.json` written by `python -m nada.download`."""
    path = path or (repo_root() / "models" / "wav2vec2-base-960h" / "vocab.json")
    if not path.is_file():
        raise FileNotFoundError(f"missing {path}. Run: pixi run convert")
    return {int(index): token for token, index in json.loads(path.read_text()).items()}


Greedy CTC: take the best token per frame, merge runs of the same token, then drop the blank (`<pad>`). A blank between two equal tokens keeps both, which is how double letters survive. `|` is the word boundary.


In [ ]:
#| export
_SPECIAL = {"<pad>", "<s>", "</s>", "<unk>"}


In [ ]:
#| export
def ctc_decode(ids: np.ndarray, vocab: dict[int, str]) -> list[str]:
    """Greedy-decode `(batch, frames)` token ids into one string per clip."""
    texts = []
    for row in np.asarray(ids).reshape(len(ids), -1):
        tokens, previous = [], None
        for index in row.tolist():
            if index != previous:
                tokens.append(vocab[index])
            previous = index
        pieces = [" " if token == "|" else token for token in tokens if token not in _SPECIAL]
        texts.append("".join(pieces).strip())
    return texts


In [ ]:
vocab = {0: "<pad>", 1: "<s>", 2: "</s>", 3: "<unk>", 4: "|", 5: "L", 6: "O", 7: "H", 8: "I"}
frames = np.array([[7, 7, 8, 0, 0, 4, 5, 5, 0, 5, 6, 6, 3, 0]])
assert ctc_decode(frames, vocab) == ["HI LLO"]


In [ ]:
clip = np.random.default_rng(0).standard_normal(1000) * 3 + 2
out = normalize(clip)
assert abs(out.mean()) < 1e-5 and abs(out.std() - 1) < 1e-3
batch = prepare([clip, clip[:600]], Wav2Vec2Config())
assert batch.shape == (2, 1000) and not batch[1, 600:].any()
